# Lab Exercise: Building a Validated REST API for ML Model Serving
## AIAT 125 — Unit 2: Packaging and Serving AI Models

**Learning objectives**
1. Build a Flask REST API with proper input validation around a trained sklearn model.
2. Use Flask's `test_client()` to write automated tests without starting a live server.
3. Benchmark p50/p95 latency across 50 requests.
4. Add a `/health` endpoint and test it.

**Why this matters**  
A model in a notebook is useless to a mobile app or a dashboard. An HTTP API is the universal interface — any language can send a JSON request and receive a JSON response. The model stays in one place; the world calls it.

**Grading** — 100 points total
| Task | Points |
|---|---|
| Task 1: `/predict` endpoint with validation | 30 |
| Task 2: Automated test suite (4 cases) | 25 |
| Task 3: Latency benchmark p50/p95 | 20 |
| Task 4: `/health` endpoint with 3 required fields | 25 |


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "flask", "scikit-learn", "joblib"], check=False)

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib, numpy as np, json, time, os

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

MODEL_PATH = "/tmp/iris_rf_exercise.joblib"
joblib.dump(clf, MODEL_PATH)
print(f"Model saved: accuracy = {clf.score(X_test, y_test):.2%}")
print("Setup complete.")

---
## Task 1 — `/predict` Endpoint with Input Validation (30 points)

Build a Flask app with a `/predict` endpoint. It must:
1. Accept POST requests with a JSON body containing: `sepal_length`, `sepal_width`, `petal_length`, `petal_width`
2. Return **400** if body is not valid JSON or is empty
3. Return **422** if any of the 4 keys is missing
4. Return **422** if any value cannot be converted to float
5. Return **200** with `{"prediction": "setosa", "confidence": 0.98}` on valid input

HTTP status semantics:
- `400 Bad Request` — malformed body (can't parse it)
- `422 Unprocessable Entity` — valid JSON but semantically wrong (missing fields, wrong types)
- `200 OK` — success

> **Important**: Also define a `/health` endpoint placeholder here — you'll complete it in Task 4.

In [ ]:
from flask import Flask, request, jsonify
import joblib, numpy as np

app = Flask(__name__)

CLASS_NAMES = ["setosa", "versicolor", "virginica"]
REQUIRED_KEYS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
_model = joblib.load(MODEL_PATH)
_app_version = "1.0.0"

@app.route("/health")
def health():
    # TODO Task 4: return {"status": "ok", "model_loaded": True, "version": "1.0.0"}
    # For now, return a minimal placeholder so the app runs
    return jsonify({"status": "placeholder"})

@app.route("/predict", methods=["POST"])
def predict():
    # TODO 1a: Parse JSON body using request.get_json(silent=True)
    # If result is None or not a dict, return 400 with error message
    data = None  # YOUR CODE: replace with request.get_json(silent=True)
    # if data is None or not isinstance(data, dict):
    #     return jsonify({"error": "Request body must be a JSON object"}), 400

    # TODO 1b: Check all REQUIRED_KEYS are present.
    # Collect missing = [k for k in REQUIRED_KEYS if k not in data]
    # If missing, return 422 with {"error": f"Missing fields: {missing}"}

    # TODO 1c: Convert values to float.
    # try: features = [float(data[k]) for k in REQUIRED_KEYS]
    # except (ValueError, TypeError): return 422 with error
    features = None  # YOUR CODE

    # TODO 1d: Run prediction
    # arr = np.array([features])
    # proba = _model.predict_proba(arr)[0]
    # class_idx = int(np.argmax(proba))
    # return jsonify({"prediction": CLASS_NAMES[class_idx],
    #                 "confidence": round(float(proba[class_idx]), 4)})
    pass  # YOUR CODE

print("Flask app defined.")

---
## Task 2 — Automated Test Suite (25 points)

Flask's `test_client()` calls endpoints in-process — no live server needed. Write 4 tests:

| Test | Input | Expected status | Check |
|---|---|---|---|
| A: Happy path | sepal=5.1×3.5, petal=1.4×0.2 | 200 | `prediction` and `confidence` keys present |
| B: Missing field | only 3 of 4 keys | 422 | — |
| C: Wrong type | `sepal_length="five"` | 422 | — |
| D: Empty body | empty string | 400 | — |

In [ ]:
client = app.test_client()

# --- Test A: Happy path ---
valid_payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
# TODO: send POST to /predict with valid_payload
# Check: status == 200, 'prediction' in body, 'confidence' in body
resp_a = None  # YOUR CODE: client.post("/predict", ...)
body_a = None  # YOUR CODE: json.loads(resp_a.data)
test_a_passed = None  # YOUR CODE: resp_a.status_code == 200 and ...

# --- Test B: Missing field ---
partial_payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4}  # no petal_width
# TODO: send POST and check status == 422
resp_b = None  # YOUR CODE
test_b_passed = None  # YOUR CODE: resp_b.status_code == 422

# --- Test C: Wrong type ---
bad_type_payload = {"sepal_length": "five", "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
# TODO: send POST and check status == 422
resp_c = None  # YOUR CODE
test_c_passed = None  # YOUR CODE

# --- Test D: Empty body ---
# TODO: send POST with data="", content_type="application/json" and check status == 400
resp_d = None  # YOUR CODE
test_d_passed = None  # YOUR CODE

# --- Print results ---
results = {
    "A: happy path (expect 200)"  : test_a_passed,
    "B: missing field (expect 422)": test_b_passed,
    "C: wrong type (expect 422)"  : test_c_passed,
    "D: empty body (expect 400)"  : test_d_passed,
}
print("=== API Test Suite ===")
all_passed = True
for name, passed in results.items():
    if passed is None:
        print(f"  [SKIP] {name}")
        all_passed = False
    else:
        print(f"  [{'PASS' if passed else 'FAIL'}] {name}")
        if not passed:
            all_passed = False

assert all_passed, "Some tests failed or were not implemented"
print("\nTask 2 PASSED")

---
## Task 3 — Latency Benchmark (20 points)

Send 50 requests using `test_client()`. Measure each with `time.perf_counter()` and compute p50/p95.

> `test_client()` latency is in-process (no network). Real HTTP adds ~1–5 ms. These numbers = pure model inference + serialization cost.

In [ ]:
payload_str = json.dumps({"sepal_length": 5.1, "sepal_width": 3.5,
                           "petal_length": 1.4, "petal_width": 0.2})

# TODO: Run 50 POST requests to /predict
# Measure each with time.perf_counter() and store latency in milliseconds
bench_latencies = []  # do NOT rename
# YOUR CODE HERE

# TODO: Compute p50 and p95
bench_p50 = None  # np.percentile(bench_latencies, 50)
bench_p95 = None  # np.percentile(bench_latencies, 95)

print(f"Benchmark (50 requests):")
print(f"  p50 (median) : {bench_p50:.2f} ms" if bench_p50 else "  p50: not computed")
print(f"  p95          : {bench_p95:.2f} ms" if bench_p95 else "  p95: not computed")

assert len(bench_latencies) == 50, f"Need 50 latencies, got {len(bench_latencies)}"
assert bench_p50 is not None and bench_p95 is not None
assert bench_p95 < 500, f"p95 should be < 500ms in-process, got {bench_p95:.1f}ms"
print("Task 3 PASSED")

---
## Task 4 — `/health` Endpoint (25 points)

Load balancers and Kubernetes health checks call `/health` to verify the service is alive and ready. Without this, traffic cannot be routed to the pod.

**Step 1**: Go back to the Task 1 cell and replace the placeholder `/health` implementation with:
```python
return jsonify({"status": "ok", "model_loaded": True, "version": "1.0.0"})
```
**Step 2**: Re-run the Task 1 cell.
**Step 3**: Run the assertions below.

In [ ]:
# TODO: After updating /health in Task 1 and re-running that cell, test it here
resp_health = client.get("/health")
body_health = json.loads(resp_health.data)

print(f"Health status code : {resp_health.status_code}")
print(f"Health body        : {body_health}")

# TODO: Write 3 assertions:
# 1. resp_health.status_code == 200
# 2. body_health["status"] == "ok"
# 3. body_health["model_loaded"] == True
# YOUR CODE HERE

print("Task 4 PASSED")

In [ ]:
# --- Final summary ---
print("=" * 55)
print("UNIT 2 LAB — FINAL DEPLOYMENT GATE")
print("=" * 55)

checks = {
    "Task 1: /predict returns 200 on valid input" :
        resp_a is not None and resp_a.status_code == 200,
    "Task 2: all 4 test cases pass" : all_passed,
    "Task 3: 50-request benchmark p95 < 500ms" :
        bench_p95 is not None and bench_p95 < 500,
    "Task 4: /health has status/model_loaded/version" :
        resp_health.status_code == 200 and
        all(k in body_health for k in ["status", "model_loaded", "version"]),
}
for task, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {task}")
print("=" * 55)

---
## Self-Check Questions

1. **Why return 422 for a missing field instead of 400?** What is the semantic difference?
2. **Why load the model at module level instead of inside `/predict`?** What happens to latency if the model reloads on every request?
3. **The p95 latency from `test_client()` is lower than real HTTP. What adds latency in production?** Name 3 sources.
4. **If `/health` always returns 200 even when the model is not loaded, what happens to Kubernetes routing?** Why is checking `model_loaded` critical?